# 13 — SAM2 Boundary Refinement on DINOv3 Field Boundaries

Standalone SAM2 boundary refinement on pre-computed DINOv3 field boundaries.
Each polygon's bounding box is fed to SAM2 as a prompt, producing pixel-accurate
masks that replace the original geometry.

**Input:** `fields_presam_sentinel2_dinov3_2020.gpkg` (DINOv3 fields)
**Raster:** `sentinel2_2020_composite.tif` (Sentinel-2 annual composite)

**Prerequisites:**
```bash
pip install agribound[samgeo]
```

## Configuration

In [ ]:
import logging
from pathlib import Path

import geopandas as gpd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(message)s",
    datefmt="%H:%M:%S",
)
logging.getLogger("root").setLevel(logging.WARNING)
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("agribound").setLevel(logging.INFO)

OUTPUT_DIR = Path("../../outputs/lea_county_ensemble")
INPUT_GPKG = OUTPUT_DIR / "fields_sentinel2_dinov3_2020.gpkg"
RASTER_PATH = OUTPUT_DIR / ".agribound_cache" / "sentinel2_2020_composite.tif"
OUTPUT_GPKG = OUTPUT_DIR / "fields_sam2_sentinel2_dinov3_2020.gpkg"
OUTPUT_CRS = "EPSG:26913"

SAM_MODEL = "large"
SAM_BATCH_SIZE = 100
SMOOTH_ITERATIONS = 3
SIMPLIFY_TOLERANCE = 2.0

## Load Input Boundaries

In [ ]:
gdf = gpd.read_file(INPUT_GPKG)
print(f"Loaded {len(gdf)} field boundaries from {INPUT_GPKG}")
print(f"CRS: {gdf.crs}")

## Run SAM2 Refinement

In [ ]:
import time

from agribound.config import AgriboundConfig
from agribound.engines.samgeo_engine import refine_boundaries

config = AgriboundConfig(
    source="sentinel2",
    engine="dinov3",
    year=2020,
    study_area="",
    output_path=str(OUTPUT_GPKG),
    engine_params={
        "sam_model": SAM_MODEL,
        "sam_batch_size": SAM_BATCH_SIZE,
    },
    device="auto",
)

print(f"Running SAM2 refinement (model={SAM_MODEL})")
tic = time.time()
refined = refine_boundaries(gdf, str(RASTER_PATH), config)
elapsed = time.time() - tic
print(f"SAM2 refined {len(refined)} boundaries in {elapsed:.1f}s")

## Post-process and Save

In [ ]:
from agribound.postprocess.simplify import simplify_polygons, smooth_polygons

refined = refined[~refined.geometry.is_empty].copy()
refined = refined[refined.geometry.is_valid].copy()
refined = refined[refined.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
refined = refined.reset_index(drop=True)
print(f"{len(gdf) - len(refined)} fields removed due to invalid geometry")

refined = smooth_polygons(refined, iterations=SMOOTH_ITERATIONS)
refined = simplify_polygons(refined, tolerance=SIMPLIFY_TOLERANCE)

if refined.crs is not None and str(refined.crs) != OUTPUT_CRS:
    refined = refined.to_crs(OUTPUT_CRS)

if OUTPUT_GPKG.exists():
    OUTPUT_GPKG.unlink()
refined.to_file(OUTPUT_GPKG, driver="GPKG", layer="fields")
print(f"Saved to {OUTPUT_GPKG}")

## Before / After Comparison

In [ ]:
gdf_m = gdf.to_crs(OUTPUT_CRS) if gdf.crs and gdf.crs.is_geographic else gdf
ref_m = refined.to_crs(OUTPUT_CRS) if refined.crs and refined.crs.is_geographic else refined

area_before = gdf_m.geometry.area.sum() / 10000
area_after = ref_m.geometry.area.sum() / 10000

print(f"{'Metric':<20} {'Before':>10} {'After':>10}")
print(f"{'-' * 20} {'-' * 10} {'-' * 10}")
print(f"{'Fields':<20} {len(gdf):>10} {len(refined):>10}")
print(f"{'Total area (ha)':<20} {area_before:>10,.1f} {area_after:>10,.1f}")

## Evaluate Against NMOSE Reference

In [ ]:
nmose_path = Path("../../examples/NMOSE Field Boundaries/WUCB ag polys.shp")
if nmose_path.exists():
    from agribound.evaluate import evaluate

    ref_gdf = gpd.read_file(nmose_path)
    ref_county = ref_gdf[ref_gdf["County"] == "25"].copy()

    print(f"Evaluation against NMOSE reference ({len(ref_county)} polygons):")
    print(f"{'Metric':<20} {'Before':>10} {'After':>10}")
    print(f"{'-' * 20} {'-' * 10} {'-' * 10}")

    try:
        m_before = evaluate(gdf, ref_county)
        m_after = evaluate(refined, ref_county)
        for key in ["f1", "iou_mean", "precision", "recall"]:
            print(f"{key:<20} {m_before[key]:>10.3f} {m_after[key]:>10.3f}")
    except Exception as exc:
        print(f"  Evaluation failed: {exc}")
else:
    print("NMOSE reference shapefile not found, skipping evaluation.")